In [ ]:
from utils import *
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
from loaders import baseline_loader

In [ ]:
res_df, scalars = baseline_loader()

In [ ]:
from itertools import islice

envs = res_df["env"].unique()
ratios = res_df["ratio"].unique()

fig = make_subplots(
    cols=len(envs),
    column_titles=[*envs],
)

val_ranges = {
    "Assault": [1.5, 5.0],
    "CrazyClimber": [0.5, 1.5],
    "MsPacman": [1.2, 2.5],
}

colors = [*islice(make_color_iter(), 0, len(ratios))]
for col, env in enumerate(envs, 1):
    env_dfs = res_df[res_df["env"] == env].copy()

    for idx, ratio in enumerate(ratios):
        tests = env_dfs[env_dfs["ratio"] == ratio]
        # df = scalars.read(tests[tests["seed"] == 0].iloc[0]["path"])
        # df = df[df["tag"] == "wm/val_loss"].copy()
        dfs = []
        for _, test in tests.iterrows():
            test_df = scalars.read(test["path"])
            test_df = test_df[test_df["tag"] == "wm/val_loss"].copy()
            dfs.append(test_df)
        dfs = pd.concat(dfs)
        g = dfs.groupby("step")
        df = pd.concat({"value": g["value"].mean()})
        df = df.reset_index()
        df["step"] *= ratio
        df["value"] = exp_mov_avg(df["value"], 1.0 - 5e-3 * ratio)
        fig.add_trace(
            go.Scatter(
                x=df["step"],
                y=df["value"],
                showlegend=(col == 1),
                legendgroup=f"{ratio}",
                name=f"Ratio = {ratio}",
                line=dict(color=colors[idx]),
            ),
            row=1,
            col=col,
        )

    fig.update_layout(**{f"yaxis{col}": dict(range=val_ranges[env])})

fig

In [ ]:
envs = res_df["env"].unique()

fig = make_subplots(
    rows=1,
    cols=len(envs),
    column_titles=[*envs],
)

for col, env in enumerate(envs, 1):
    score_df = res_df[res_df["env"] == env].copy()

    final_values = []
    for idx, row in score_df.iterrows():
        test_df = scalars.read(row["path"])
        values = test_df[test_df["tag"] == "val/wm_loss"]["value"].to_numpy()
        # val_loss = exp_mov_avg(val_loss, 0.9)
        final_values.append(values[-1])

    score_df["wm_loss"] = final_values

    subplot = px.scatter(score_df, x="score", y="wm_loss", trendline="ols")
    for trace in subplot.data:
        fig.add_trace(trace, row=1, col=col)

fig.update_layout(width=1000, height=400)

fig.write_image("../tex/assets/model_val_loss.pdf")
fig

In [ ]:
res_df

In [ ]:
envs = res_df["env"].unique()

fig = make_subplots(
    rows=1,
    cols=len(envs),
    column_titles=[*envs],
)

for col, env in enumerate(envs, 1):
    score_df = res_df[res_df["env"] == env].copy()

    final_values = []
    for idx, row in score_df.iterrows():
        test_df = scalars.read(row["path"])
        values = test_df[test_df["tag"] == "val/wm_loss"]["value"].to_numpy()
        # val_loss = exp_mov_avg(val_loss, 0.9)
        final_values.append(values[-1])

    score_df["wm_loss"] = final_values
    subplot = px.scatter(score_df, x="score", y="wm_loss", trendline="ols")
    for trace in subplot.data:
        fig.add_trace(trace, row=1, col=col)

fig.update_layout(width=1000, height=400)

fig

In [ ]:
envs = res_df["env"].unique()

fig = make_subplots(
    rows=1,
    cols=len(envs),
    column_titles=[*envs],
)

for col, env in enumerate(envs, 1):
    score_df = res_df[res_df["env"] == env].copy()

    final_values = []
    for idx, row in score_df.iterrows():
        test_df = scalars.read(row["path"])
        values = test_df[test_df["tag"] == "rl/entropy"]["value"].to_numpy()
        final_values.append(values[-1])

    score_df["entropy"] = final_values
    subplot = px.scatter(score_df, x="score", y="entropy", trendline="ols")
    for trace in subplot.data:
        fig.add_trace(trace, row=1, col=col)

fig.update_layout(width=1000, height=400)

fig